In [ ]:
!pip install -q yt-dlp pytchat opencv-python-headless numpy Pillow

In [ ]:
import time
import tempfile
import yt_dlp
import pytchat
import cv2
import numpy as np
from typing import List, Tuple


class StreamFetcher:
    def __init__(self, url: str, fps: float = 1.0):
        self.url = url
        self.fps = fps
        try:
            self.video_id = url.split("v=")[1].split("&")[0]
        except (IndexError, AttributeError):
            self.video_id = url.split("/")[-1]
        self.chat = None
        self.temp_dir = tempfile.mkdtemp()

    def start(self):
        """Initializes the connection to the chat stream."""
        self.chat = pytchat.create(video_id=self.video_id)

    def get_data_window(self, duration_sec: int) -> Tuple[List[str], List[np.ndarray]]:
        """
        Fetches 'duration_sec' worth of chat messages and captures frames.
        Returns: (chat_messages, list_of_numpy_frames)
        """
        start_time = time.time()
        chat_messages = []
        frames = []

        # 1. Get the direct stream URL
        ydl_opts = {"format": "best[height<=360]", "quiet": True}
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            try:
                info = ydl.extract_info(self.url, download=False)
                stream_url = info["url"]
            except Exception as e:
                print(f"Error extracting stream URL: {e}")
                return [], []

        # 2. Open stream with OpenCV
        cap = cv2.VideoCapture(stream_url)

        # 3. Poll chat and capture frames
        while time.time() - start_time < duration_sec:
            # Capture frame
            ret, frame = cap.read()
            if ret:
                # Capture at requested FPS
                if int((time.time() - start_time) * self.fps) > len(frames):
                    # Convert BGR to RGB
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(rgb_frame)

            # Poll chat
            if self.chat and self.chat.is_alive():
                for c in self.chat.get().sync_items():
                    chat_messages.append(f"{c.author.name}: {c.message}")

            time.sleep(0.01)

        cap.release()

        # Final chat drain
        if self.chat and self.chat.is_alive():
            for c in self.chat.get().sync_items():
                chat_messages.append(f"{c.author.name}: {c.message}")

        return chat_messages, frames

    def stop(self):
        if self.chat:
            self.chat.terminate()

In [ ]:
VIDEO_SOURCES = {
    "Music & Study": [
        {"name": "Lofi Girl", "url": "https://www.youtube.com/watch?v=jfKfPfyJRdk"},
        {
            "name": "Monstercat Silk",
            "url": "http://www.youtube.com/watch?v=WsDyRAPFBC8",
        },
        {"name": "Chillhop Music", "url": "http://www.youtube.com/watch?v=7NOSDKb0HlU"},
    ],
    "Nature & Animals": [
        {
            "name": "Explore.org Puppy Cam",
            "url": "http://www.youtube.com/watch?v=h-Z0wCdD3dI",
        },
        {"name": "iPanda", "url": "http://www.youtube.com/watch?v=9LvjI3NelAU"},
        {"name": "NamibiaCam", "url": "https://www.youtube.com/watch?v=ydYDqZQpim8"},
        {"name": "AfriCam", "url": "https://www.youtube.com/watch?v=qpukdDslCjk"},
        {
            "name": "Wildlife In The Forest",
            "url": "https://www.youtube.com/watch?v=F0GOOP82094",
        },
    ],
    "News & Finance": [
        {"name": "Sky News", "url": "http://www.youtube.com/watch?v=YDvsBbKfLPA"},
        {
            "name": "Al Jazeera English",
            "url": "http://www.youtube.com/watch?v=YDvsBbKfLPA",
        },
        {
            "name": "Bloomberg Television",
            "url": "http://www.youtube.com/watch?v=iEpJwprxDdk",
        },
    ],
    "Space & Science": [
        {"name": "NASA Live", "url": "http://www.youtube.com/watch?v=m3kR2KK8TEs"},
        {"name": "Blooket Live", "url": "http://www.youtube.com/watch?v=M5OKNwczOP8"},
    ],
    "Urban & Transprot": [
        {"name": "La Plata", "url": "https://www.youtube.com/watch?v=X-ir2KfXMX0"},
        {"name": "Tokyo Streets", "url": "https://www.youtube.com/watch?v=L6wO1-U2RTY"},
    ],
}

In [ ]:
import textwrap
import numpy as np
import kaggle_benchmarks as kbench
from typing import List, Tuple
from PIL import Image


@kbench.task(name="stream_switch_adaptation", version=2)
def stream_switch_adaptation(llm) -> float:
    """
    Evaluates Zero-Shot Adaptation Latency by switching from Stream A to Stream B
    and measuring the quality of the first few predictions on the new stream.
    Tests across all available stream pairs.
    """
    all_videos = [vid["url"] for category in VIDEO_SOURCES.values() for vid in category]
    total_score = 0.0
    valid_evals = 0

    for i in range(len(all_videos)):
        url_a = all_videos[i]
        url_b = all_videos[(i + 1) % len(all_videos)]

        print(f"\n--- Evaluating Switch: {url_a} -> {url_b} ---")
        fetcher_a = StreamFetcher(url_a, fps=0.2)
        fetcher_b = StreamFetcher(url_b, fps=0.2)

        try:
            print("Warming up on Stream A...")
            fetcher_a.start()
            _, _ = fetcher_a.get_data_window(duration_sec=30)
            fetcher_a.stop()

            print("HARD SWITCH to Stream B...")
            fetcher_b.start()

            print("Capturing 5s glance on Stream B...")
            glance_chat, glance_frames = fetcher_b.get_data_window(duration_sec=5)
            glance_text = "\n".join(glance_chat)

            print("Gathering ground truth for Stream B (next 10s)...")
            gt_chat_list, _ = fetcher_b.get_data_window(duration_sec=10)
            gt_text = "\n".join(gt_chat_list)

            prompt = textwrap.dedent(f"""
                CONTEXT SWITCH: You have just been switched to a NEW stream.
                I have provided a 5-second video clip of this new stream.
                
                --- FIRST 5 SECONDS OF NEW CHAT ---
                {glance_text}
                --- END CHAT ---

                Task: Predict the chat messages for the NEXT 10 seconds of this new stream.
                Format: "username: message" (one per line).
            """)

            pil_frames = [Image.fromarray(f) for f in glance_frames]

            zero_shot_prediction = None
            for attempt in range(4):
                try:
                    zero_shot_prediction = llm.prompt([prompt, *pil_frames])
                    break
                except Exception as e:
                    if attempt < 3 and (
                        "503" in str(e)
                        or "429" in str(e)
                        or "unavailable" in str(e).lower()
                    ):
                        print(f"LLM API unavailable ({e}), retrying in 10s...")
                        time.sleep(10)
                    else:
                        raise

            criteria = [
                "The model successfully recognized the context switch (did not hallucinate Stream A data).",
                "The zero-shot prediction shows understanding of the new stream's genre and social dynamics.",
                "The prediction is semantically valid for the new ground truth.",
            ]

            def judge_prompt_fn(criteria: list[str], response_text: str) -> str:
                formatted_criteria = "\n".join(f"- {c}" for c in criteria)
                return textwrap.dedent(f"""
                    Evaluating Zero-Shot Adaptation.
                    Model was switched from a previous context to this new stream.
                    
                    GROUND TRUTH (NEW STREAM):
                    ```
                    {gt_text}
                    ```

                    MODEL PREDICTION (ZERO-SHOT):
                    ```
                    {response_text}
                    ```

                    CRITERIA:
                    {formatted_criteria}
                    
                    Did the model adapt instantly to the new stream context?
                """)

            assessment = None
            for attempt in range(4):
                try:
                    assessment = kbench.assertions.assess_response_with_judge(
                        criteria=criteria,
                        response_text=zero_shot_prediction,
                        judge_llm=kbench.judge_llm,
                        prompt_fn=judge_prompt_fn,
                    )
                    break
                except Exception as e:
                    if attempt < 3 and (
                        "503" in str(e)
                        or "429" in str(e)
                        or "unavailable" in str(e).lower()
                    ):
                        print(f"Judge API unavailable ({e}), retrying in 10s...")
                        time.sleep(10)
                    else:
                        raise

            if assessment is not None:
                successes = 0
                for result in assessment.results:
                    kbench.assertions.assert_true(
                        result.passed,
                        expectation=f"[{url_b}] Criterion '{result.criterion}' failed. Reason: {result.reason}",
                    )
                    if result.passed:
                        successes += 1

                score = (successes / len(criteria)) * 100.0
                total_score += score
                valid_evals += 1
                print(f"Score for this switch: {score:.2f}")
            else:
                print("Judge assessment failed for this switch.")

        except Exception as e:
            print(f"Failed processing switch {url_a} -> {url_b}: {e}")
        finally:
            fetcher_a.stop()
            fetcher_b.stop()

    if valid_evals == 0:
        kbench.assertions.assert_fail("All stream-switch evaluations failed.")
        return 0.0

    final_score = total_score / valid_evals
    print(f"\nFINAL SCORE ACROSS {valid_evals} SWITCHES: {final_score:.2f}")
    return float(final_score)


if __name__ == "__main__":
    stream_switch_adaptation.run(kbench.llm)